# Code mẫu — Augmentation và transfer learning

Biến đổi ảnh để có thêm dữ liệu chỉ hợp lệ khi nhãn được biến đổi theo cho đúng nghĩa. Khóa encoder chỉ có tác dụng khi ta kiểm cả tham số lẫn các thống kê nó giữ bên trong. Ba ví dụ dưới đây kiểm những điều kiện đó trên dữ liệu nhỏ, trước khi đụng tới một model lớn.

## Ví dụ 1: Lật ảnh thì phải lật cả box và mask

Ảnh rộng 8 pixel, có một vùng sáng ở hàng 1–2, cột 1–3. Box được ghi theo dạng `xyxy = [1, 1, 4, 3]`, tức tọa độ mép trái, mép trên, mép phải, mép dưới. Theo trục x, box phủ đoạn từ 1 tới 4, gồm ba cột 1, 2, 3. Lật ngang ảnh thì vùng sáng chuyển sang cột 4–6, và box phải thành `[4, 1, 7, 3]`.

In [ ]:
import numpy as np

mask = np.zeros((4, 8), dtype=np.uint8)
mask[1:3, 1:4] = 1
image = 30 + 180 * mask.astype(np.int16)
box = np.array([1, 1, 4, 3])

def flip_box(box, width):
    x1, y1, x2, y2 = box
    return np.array([width - x2, y1, width - x1, y2])

flipped_image = np.flip(image, axis=1).copy()
flipped_mask = np.flip(mask, axis=1).copy()
flipped_box = flip_box(box, image.shape[1])
print("Mask trước:")
print(mask)
print("Mask sau:")
print(flipped_mask)
print("Box sau:", flipped_box.tolist())
print("Diện tích trước/sau:", int(mask.sum()), int(flipped_mask.sum()))
assert np.array_equal(flipped_image > 100, flipped_mask == 1)
assert np.array_equal(flip_box(flipped_box, 8), box)
assert np.array_equal(np.flip(flipped_mask, axis=1), mask)

Mask trước:
[[0 0 0 0 0 0 0 0]
 [0 1 1 1 0 0 0 0]
 [0 1 1 1 0 0 0 0]
 [0 0 0 0 0 0 0 0]]
Mask sau:
[[0 0 0 0 0 0 0 0]
 [0 0 0 0 1 1 1 0]
 [0 0 0 0 1 1 1 0]
 [0 0 0 0 0 0 0 0]]
Box sau: [4, 1, 7, 3]
Diện tích trước/sau: 6 6


Sau khi lật, mask vẫn có sáu pixel, và vẫn nằm đúng dưới phần sáng của ảnh lật. Ba dòng assert kiểm ba điều: mask khớp ảnh, lật box hai lần thì về chỗ cũ, lật mask hai lần cũng về chỗ cũ.

Có hai công thức dễ nhầm. Với mép của box, vị trí `x` lật thành `W − x`. Với chỉ số cột của pixel, cột `c` lật thành `W − 1 − c`. Dùng nhầm công thức này cho trường hợp kia, nhãn sẽ lệch một pixel so với ảnh.

## Ví dụ 2: Resize mask, và nhãn có hướng

Mask ở đây chứa mã lớp 0 và 2, không phải độ sáng. Ta phóng to mask theo hai cách nội suy để xem cách nào giữ đúng mã lớp. Sau đó lật một mũi tên chỉ sang trái, để thấy vì sao nhãn “trái” không thể giữ nguyên khi lật ngang.

In [ ]:
import torch
import torch.nn.functional as F

torch.set_num_threads(1)
mask = torch.tensor([[[[0., 2.], [0., 2.]]]])
nearest = F.interpolate(mask, size=(2, 4), mode="nearest")
bilinear = F.interpolate(mask, size=(2, 4), mode="bilinear", align_corners=False)
print("Nearest, hàng đầu:", nearest[0, 0, 0].tolist())
print("Bilinear, hàng đầu:", bilinear[0, 0, 0].tolist())
assert set(nearest.unique().tolist()) <= {0.0, 2.0}
assert 0.5 in bilinear.unique().tolist()

left_arrow = torch.tensor([[0, 1, 0, 0], [1, 1, 1, 1], [0, 1, 0, 0]])
right_arrow = left_arrow.flip(1)
label_after_flip = {"trái": "phải", "phải": "trái"}["trái"]
print("Mũi tên sau lật:", right_arrow.tolist())
print("Nhãn sau lật:", label_after_flip)

Nearest, hàng đầu: [0.0, 0.0, 2.0, 2.0]
Bilinear, hàng đầu: [0.0, 0.5, 1.5, 2.0]
Mũi tên sau lật: [[0, 0, 1, 0], [1, 1, 1, 1], [0, 0, 1, 0]]
Nhãn sau lật: phải


Nội suy nearest chỉ chép lại giá trị gần nhất, nên mask vẫn chỉ có 0 và 2. Nội suy bilinear lấy trung bình có trọng số, và tạo ra 0.5, 1.5, là những mã lớp không tồn tại. Ảnh thì resize bằng bilinear được, nhưng mask phải dùng nearest.

Lật ngang mũi tên chỉ trái thì được một mũi tên chỉ phải. Nếu bài yêu cầu đoán hướng, nhãn phải đổi theo. Nếu bài chỉ hỏi “có mũi tên hay không”, nhãn giữ nguyên. Một phép augmentation có hợp lệ hay không là tùy câu hỏi mà model phải trả lời.

## Ví dụ 3: Học lớp đầu ra, và phát hiện thống kê vẫn bị đổi

Encoder nhỏ ở đây được khởi tạo ngẫu nhiên, không phải model pretrained. Ta chỉ dùng nó để kiểm cơ chế: encoder ở chế độ eval, mỗi ảnh được biến thành bốn đặc trưng, và optimizer chỉ cập nhật lớp đầu ra.

In [ ]:
import torch
from torch import nn

torch.set_num_threads(1)
torch.manual_seed(53)
encoder = nn.Sequential(nn.Conv2d(1, 4, 3, padding=1), nn.BatchNorm2d(4),
                        nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten(1))
head = nn.Linear(4, 2)
images = torch.stack([torch.zeros(1, 6, 6), torch.ones(1, 6, 6)] * 3)
labels = torch.tensor([0, 1] * 3)
for p in encoder.parameters():
    p.requires_grad_(False)
encoder.eval()
optimizer = torch.optim.SGD(head.parameters(), lr=0.5)
before = {name: value.clone() for name, value in encoder.state_dict().items()}
head_before = head.weight.detach().clone()
with torch.no_grad():
    features = encoder(images)
for step in range(20):
    optimizer.zero_grad()
    loss = nn.functional.cross_entropy(head(features), labels)
    loss.backward()
    optimizer.step()
print("Features:", tuple(features.shape))
print("Encoder cả parameter/buffer giữ nguyên:",
      all(torch.equal(value, encoder.state_dict()[name]) for name, value in before.items()))
print("Head đã đổi:", not torch.equal(head_before, head.weight))
assert all(p.grad is None for p in encoder.parameters())
assert not torch.equal(head_before, head.weight)

Features: (6, 4)
Encoder cả parameter/buffer giữ nguyên: True
Head đã đổi: True


Shape `(6, 4)` nghĩa là sáu ảnh, mỗi ảnh bốn đặc trưng. Ta so toàn bộ `state_dict` của encoder trước và sau. Cách so này kiểm cả trọng số conv lẫn các thống kê của BatchNorm, chặt hơn việc chỉ so trọng số. Encoder giữ nguyên, còn lớp đầu ra đã học.

Giờ ta cố tình chuyển encoder sang chế độ train, để tái hiện một lỗi rất hay gặp. Các tham số vẫn bị khóa như cũ.

In [ ]:
running_before = encoder[1].running_mean.clone()
encoder.train()
with torch.no_grad():
    encoder(images)
print("Parameter còn đóng băng:", all(not p.requires_grad for p in encoder.parameters()))
print("Running mean đã đổi:", not torch.equal(running_before, encoder[1].running_mean))
encoder.eval()
assert not torch.equal(running_before, encoder[1].running_mean)

Parameter còn đóng băng: True
Running mean đã đổi: True


Hai dòng `True` cùng đúng một lúc. Tham số vẫn bị khóa, nhưng running mean của BatchNorm đã đổi, chỉ sau một lần forward ở chế độ train. Khóa tham số không đủ để giữ encoder đứng yên. Phải đặt nó ở chế độ eval. Khi làm với một model pretrained thật, ta còn phải nạp đúng trọng số và dùng đúng cách tiền xử lý của model, rồi mới kiểm theo quy trình như trên.